# VQA Autopilot — Final Pipeline
## Qwen2.5-VL-7B + YOLO + Advanced Sampling + JSON Output + Checkpointing

```
VIDEO
  ↓
[Advanced Frame Sampler]
  ↓
┌─────────────────────────────────────────┐
│ 1. YOLO (GPU 0)   → entity detection    │  parallel
│ 2. Qwen2.5-VL-7B  → JSON answer output  │  parallel
└─────────────────────────────────────────┘
          ↓
   Feature Fusion into Qwen prompt
          ↓
   JSON parse (4-stage fallback)
          ↓
   Rule Consistency Layer
          ↓
   Checkpoint every 50 → Final CSV
```


In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "No GPU detected")

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU {i} : {p.name}  ({p.total_memory//1024**3}GB total | {free/1024**3:.1f}GB free)")


Wed Apr  8 20:55:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%capture
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q ultralytics
!pip install -q transformers>=4.45.0
!pip install -q accelerate
!pip install -q bitsandbytes
!pip install -q qwen-vl-utils
!pip install -q opencv-python-headless
!pip install -q pandas pillow tqdm
print("   All dependencies installed")


In [3]:
import os
import torch
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ─── PATHS ────────────────────────────────────────────────────────────────────
VIDEO_DIR       = "/kaggle/input/datasets/shresthml/vqa-autopilot/heatmaps"
LABEL_MAP_PATH  = "/kaggle/input/datasets/shhauryajaiswal004/submission-system/label_map.json"
SAMPLE_CSV_PATH = "/kaggle/input/datasets/shhauryajaiswal004/submission-system/sample_submission.csv"
SUBMISSION_JSON = "/kaggle/working/submission.json"
SUBMISSION_CSV  = "/kaggle/working/submission_f.csv"
CHECKPOINT_DIR  = "/kaggle/working/"

# ─── MODEL ────────────────────────────────────────────────────────────────────
VLM_MODEL_NAME  = "Qwen/Qwen2.5-VL-7B-Instruct"
YOLO_MODEL_NAME = "yolov8m.pt"

# ─── SAMPLING ─────────────────────────────────────────────────────────────────
# Fixed percentiles — proven more reliable than motion-based sampling
# Weighted toward latter half where incidents peak
QWEN_PERCENTILES   = [0.10, 0.25, 0.40, 0.55, 0.70, 0.82, 0.91, 0.97]
NUM_QWEN_FRAMES    = 8
GRID_ROWS          = 2
GRID_COLS          = 4
FRAME_SIZE         = 224   
MIN_BRIGHTNESS     = 5.0     # black frame threshold
DIVERSITY_THRESH   = 0.92     # cosine similarity threshold for duplicate frames
MAX_LOAD_FRAMES    = 64       # cap raw frames loaded (speed)

# ─── PIPELINE ─────────────────────────────────────────────────────────────────
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
VIDEO_ID_RANGE = None          # None = all, (0, 100) = first 100
CHECKPOINT_EVERY = 50          # save checkpoint every N videos
QWEN_MAX_TOKENS  = 320
QWEN_USE_4BIT    = True        # 4-bit quantization for 7B on T4

print("─" * 55)
print(f"VIDEO_DIR    : {VIDEO_DIR}")
print(f"VLM_MODEL    : {VLM_MODEL_NAME}")
print(f"YOLO_MODEL   : {YOLO_MODEL_NAME}")
print(f"DEVICE       : {DEVICE}")
print(f"FRAME_SIZE   : {FRAME_SIZE}px  | FRAMES: {NUM_QWEN_FRAMES}")
print(f"PERCENTILES  : {QWEN_PERCENTILES}")
print(f"CHECKPOINT   : every {CHECKPOINT_EVERY} videos")
print("─" * 55)


───────────────────────────────────────────────────────
VIDEO_DIR    : /kaggle/input/datasets/shresthml/vqa-autopilot/heatmaps
VLM_MODEL    : Qwen/Qwen2.5-VL-7B-Instruct
YOLO_MODEL   : yolov8m.pt
DEVICE       : cuda
FRAME_SIZE   : 224px  | FRAMES: 8
PERCENTILES  : [0.1, 0.25, 0.4, 0.55, 0.7, 0.82, 0.91, 0.97]
CHECKPOINT   : every 50 videos
───────────────────────────────────────────────────────


In [4]:
import json

with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    LABEL_MAP = json.load(f)

# Valid integer values per question (for validation)
VALID_VALUES = {qid: set(qdata["answers"].values())
                for qid, qdata in LABEL_MAP.items()}

# Questions where -2 (not applicable) is NEVER valid — must stay -1
NEVER_NA = {"Q1.a","Q1.b","Q2","Q3.a","Q3.b","Q4.a","Q4.b","Q4.c",
            "Q5.a","Q5.b","Q5.j","Q6.a","Q8","Q9.a","Q9.b"}

print(f"   Loaded {len(LABEL_MAP)} questions")
for qid, qdata in LABEL_MAP.items():
    n = len(qdata["answers"])
    print(f"  {qid:8s} | {n:2d} choices | {qdata['question'][:52]}...")


   Loaded 25 questions
  Q1.a     |  5 choices | What was the time of day during the incident?...
  Q1.b     |  7 choices | What were the weather conditions during the incident...
  Q2       |  5 choices | What was the lighting condition at the time of the i...
  Q3.a     |  4 choices | Where did the incident take place (traffic environme...
  Q3.b     |  6 choices | What facilities were present in the surrounding envi...
  Q4.a     |  8 choices | What is the road configuration at the incident time?...
  Q4.b     |  4 choices | Road lane configuration...
  Q4.c     |  4 choices | Lane direction of Ego-Car at time of incident...
  Q5.a     |  7 choices | What is the category of the incident?...
  Q5.b     | 10 choices | Primary Incident Entity (Who caused or could have pr...
  Q5.c     | 10 choices | If Primary Incident Entity is a vehicle, what type o...
  Q5.e     | 18 choices | Primary Entity Behavior...
  Q5.f     |  8 choices | Secondary Incident Entity (Victim, secondary cause o..

In [5]:
import cv2
import numpy as np
from typing import List, Tuple

# ─────────────────────────────────────────────────────────────
# CONFIG (assumed already defined globally)
# ─────────────────────────────────────────────────────────────
# MIN_BRIGHTNESS = 12.0
# DIVERSITY_THRESH = 0.92
# MAX_LOAD_FRAMES = 80
# NUM_QWEN_FRAMES = 8
# QWEN_PERCENTILES = [0.10, 0.25, 0.40, 0.55, 0.70, 0.82, 0.91, 0.97]

# ─────────────────────────────────────────────────────────────
# FRAME QUALITY CHECK
# ─────────────────────────────────────────────────────────────
def _is_bad_frame(frame: np.ndarray, threshold: float = MIN_BRIGHTNESS) -> bool:
    if frame is None or frame.size == 0:
        return True
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return float(gray.mean()) < threshold   # ← remove the std < 5 line


# ─────────────────────────────────────────────────────────────
# FRAME HELPERS
# ─────────────────────────────────────────────────────────────
def _read_frame_at(cap, idx, total):
    idx = int(np.clip(idx, 0, total - 1))
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if not ret or frame is None:
        return None
    return None if _is_bad_frame(frame) else frame


def _find_nearest_valid(cap, target, total, radius=15):
    for offset in range(radius + 1):
        for delta in ([offset, -offset] if offset > 0 else [0]):
            candidate = target + delta
            if 0 <= candidate < total:
                frame = _read_frame_at(cap, candidate, total)
                if frame is not None:
                    return frame
    return None


# ─────────────────────────────────────────────────────────────
# TEMPORAL COVERAGE
# ─────────────────────────────────────────────────────────────
def _ensure_temporal_coverage(frames_with_idx, cap, total):
    if total < 3 or len(frames_with_idx) == 0:
        return frames_with_idx

    current_idx = {idx for idx, _ in frames_with_idx}

    zones = [
        (0.0,  0.15, int(total * 0.08)),
        (0.40, 0.60, int(total * 0.50)),
        (0.85, 1.00, int(total * 0.92)),
    ]

    for lo, hi, target in zones:
        lo_idx = int(total * lo)
        hi_idx = int(total * hi)

        if not any(lo_idx <= i <= hi_idx for i in current_idx):
            frame = _find_nearest_valid(cap, target, total, radius=20)
            if frame is not None:
                frames_with_idx.append((target, frame))

    return sorted(frames_with_idx, key=lambda x: x[0])


# ─────────────────────────────────────────────────────────────
# DUPLICATE REMOVAL (SAFE)
# ─────────────────────────────────────────────────────────────
def _remove_duplicates(frames_with_idx, threshold=DIVERSITY_THRESH):
    if len(frames_with_idx) <= 1:
        return frames_with_idx

    thumbs = []
    for _, frame in frames_with_idx:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        t = cv2.resize(gray, (16, 16)).flatten().astype(np.float32)
        norm = np.linalg.norm(t)
        thumbs.append(t / norm if norm > 0 else t)

    duplicates = set()

    for i in range(len(thumbs)):
        if i in duplicates:
            continue
        for j in range(i + 1, len(thumbs)):
            if j in duplicates:
                continue

            sim = float(np.dot(thumbs[i], thumbs[j]))

            #  FIX: only remove near duplicates
            if abs(i - j) <= 2 and sim > threshold:
                duplicates.add(j)

    return [frames_with_idx[i] for i in range(len(frames_with_idx)) if i not in duplicates]


# ─────────────────────────────────────────────────────────────
# MAIN SAMPLER
# ─────────────────────────────────────────────────────────────
def sample_frames(video_path: str) -> Tuple[List[np.ndarray], dict]:
    meta = {"total_frames": 0, "sampled": 0, "warnings": []}

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        meta["warnings"].append("Cannot open video")
        return [], meta

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    meta["total_frames"] = total

    if total <= 0:
        cap.release()
        meta["warnings"].append("0 frames")
        return [], meta

    # Load subset
    if total <= MAX_LOAD_FRAMES:
        load_indices = list(range(total))
    else:
        load_indices = np.linspace(0, total - 1, MAX_LOAD_FRAMES).astype(int).tolist()

    loaded = {}
    for idx in load_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret and frame is not None and not _is_bad_frame(frame):
            loaded[idx] = frame

    if not loaded:
        cap.release()
        meta["warnings"].append("No valid frames")
        return [], meta

    all_idx = sorted(loaded.keys())

    # Percentile sampling
    frames_with_idx = []
    for pct in QWEN_PERCENTILES:
        target = int(pct * (total - 1))
        closest = min(all_idx, key=lambda i: abs(i - target))
        frames_with_idx.append((closest, loaded[closest]))

    # Remove duplicate indices
    seen = set()
    unique = []
    for idx, f in frames_with_idx:
        if idx not in seen:
            unique.append((idx, f))
            seen.add(idx)
    frames_with_idx = unique

    # Ensure temporal coverage
    frames_with_idx = _ensure_temporal_coverage(frames_with_idx, cap, total)

    # Remove duplicates (safe)
    frames_with_idx = _remove_duplicates(frames_with_idx)

    frames_with_idx = sorted(frames_with_idx, key=lambda x: x[0])

    # Trim / pad
    if len(frames_with_idx) > NUM_QWEN_FRAMES:
        keep = np.linspace(0, len(frames_with_idx) - 1, NUM_QWEN_FRAMES).astype(int)
        frames_with_idx = [frames_with_idx[i] for i in keep]

    elif len(frames_with_idx) < NUM_QWEN_FRAMES:
        meta["warnings"].append(f"{len(frames_with_idx)} frames → padding")

        targets = [0.2, 0.5, 0.8]

        while len(frames_with_idx) < NUM_QWEN_FRAMES:
            used = {idx for idx, _ in frames_with_idx}

            target = int(targets[len(frames_with_idx) % len(targets)] * total)
            candidates = sorted(all_idx, key=lambda i: abs(i - target))

            added = False
            for c in candidates:
                if c not in used:
                    frames_with_idx.append((c, loaded[c]))
                    frames_with_idx = sorted(frames_with_idx, key=lambda x: x[0])
                    added = True
                    break

            if not added:
                break

    cap.release()
    meta["sampled"] = len(frames_with_idx)

    return [f for _, f in frames_with_idx], meta


# ─────────────────────────────────────────────────────────────
# GRID (SAFE VERSION)
# ─────────────────────────────────────────────────────────────
def build_grid(frames, temporal, rows=2, cols=4, size=336):

    total_slots = rows * cols

    frames = frames[:total_slots]
    while len(frames) < total_slots:
        frames.append(frames[-1])

    # SAFE temporal access
    peak_phase = temporal.get("peak_phase", "incident") if isinstance(temporal, dict) else "incident"

    weights = np.ones(len(frames))

    if peak_phase == "incident":
        weights[len(frames)//3 : 2*len(frames)//3] *= 1.3
    elif peak_phase == "before":
        weights[:len(frames)//3] *= 1.3
    elif peak_phase == "after":
        weights[2*len(frames)//3:] *= 1.3

    processed = []

    for i, f in enumerate(frames):
        img = cv2.resize(f, (size, size))

        if weights[i] > 1:
            img = cv2.convertScaleAbs(img, alpha=1.15, beta=5)

        img = cv2.copyMakeBorder(img, 2, 2, 2, 2,
                                 cv2.BORDER_CONSTANT, value=(0,0,0))

        processed.append(img)

    grid_rows = []
    idx = 0

    for r in range(rows):
        row = np.hstack(processed[idx:idx+cols])
        grid_rows.append(row)
        idx += cols

    return np.vstack(grid_rows)


print("FINAL sampler + grid (fully fixed, safe, runnable)")

FINAL sampler + grid (fully fixed, safe, runnable)


In [6]:
import cv2
import numpy as np
from typing import List

def extract_temporal_features(frames: List[np.ndarray]) -> dict:
    """
    Robust temporal reasoning:
    - Handles short/static videos
    - Uses all phases (before, incident, after)
    - Stable motion ratio
    """

    if len(frames) < 2:
        return {
            "motion_before": 0.0,
            "motion_incident": 0.0,
            "motion_after": 0.0,
            "motion_ratio": 1.0,
            "is_collision": False,
            "peak_phase": "unknown",
            "intensity": "low"
        }

    n = len(frames)

    # Split into phases
    b  = frames[:max(n//3, 1)]
    mi = frames[n//3 : max(2*n//3, n//3+1)]
    a  = frames[max(2*n//3, n//3+1):]

    def motion(fs):
        if len(fs) < 2:
            return 0.0
        diffs = []
        for i in range(1, len(fs)):
            prev = cv2.cvtColor(fs[i-1], cv2.COLOR_BGR2GRAY)
            curr = cv2.cvtColor(fs[i],   cv2.COLOR_BGR2GRAY)
            diffs.append(float(np.mean(cv2.absdiff(prev, curr))))
        return float(np.mean(diffs))

    mb = motion(b)
    mm = motion(mi)
    ma = motion(a)

    #  FIX 1: Stable ratio (use max baseline)
    baseline = max(mb, ma, 1.0)
    motion_ratio = mm / baseline

    #  FIX 2: Robust peak detection
    if mm >= mb and mm >= ma:
        peak_phase = "incident"
    elif mb >= ma:
        peak_phase = "before"
    else:
        peak_phase = "after"

    #  FIX 3: Collision logic (less noisy)
    is_collision = (
        motion_ratio > 2.0 and
        mm > mb * 1.5 and
        mm > ma * 1.2
    )

    #  FIX 4: Better intensity classification
    avg_motion = (mb + mm + ma) / 3

    if avg_motion > 25:
        intensity = "high"
    elif avg_motion > 10:
        intensity = "medium"
    else:
        intensity = "low"

    return {
        "motion_before":   mb,
        "motion_incident": mm,
        "motion_after":    ma,
        "motion_ratio":    motion_ratio,
        "is_collision":    bool(is_collision),
        "peak_phase":      peak_phase,
        "intensity":       intensity,
    }

print(" Temporal branch ready (robust version)")

 Temporal branch ready (robust version)


In [7]:
from ultralytics import YOLO
import numpy as np
import torch

print("Loading YOLOv8...")

YOLO_MODEL = YOLO(YOLO_MODEL_NAME)
YOLO_MODEL.to("cuda:0")

#  REMOVE THIS (causes dtype crash)
# YOLO_MODEL.half()

#    Force model to float32 for stability
YOLO_MODEL.model.float()

print(f"   YOLOv8 loaded on cuda:0 | classes: {len(YOLO_MODEL.names)}")

VEHICLE_CLASSES = {"car","truck","bus","motorcycle","bicycle"}
PERSON_CLASSES  = {"person"}
SIGN_CLASSES    = {"stop sign","traffic light"}
ANIMAL_CLASSES  = {"dog","cat","horse","cow","sheep","bird",
                   "bear","elephant","zebra","giraffe"}


def aggregate_detections(frames: list, conf: float = 0.30) -> dict:
    """Run YOLOv8 on all frames in one batched call. Returns aggregated detections."""

    all_vehicles = set()
    all_persons  = set()
    all_signs    = set()
    all_animals  = set()
    max_veh = 0
    max_per = 0
    has_tl  = False
    has_ss  = False

    try:
        #   CRITICAL: ensure dtype is correct
        frames = [f.astype(np.uint8) for f in frames]

        with torch.no_grad():
            results = YOLO_MODEL(frames, conf=conf, verbose=False, stream=True)

        for r in results:
            frame_veh, frame_per = 0, 0

            if r.boxes is None:
                continue

            for box in r.boxes:
                label = YOLO_MODEL.names[int(box.cls[0])].lower()

                if label in VEHICLE_CLASSES:
                    all_vehicles.add(label)
                    frame_veh += 1

                elif label in PERSON_CLASSES:
                    all_persons.add(label)
                    frame_per += 1

                elif label in SIGN_CLASSES:
                    all_signs.add(label)
                    if label == "traffic light": has_tl = True
                    if label == "stop sign":     has_ss = True

                elif label in ANIMAL_CLASSES:
                    all_animals.add(label)

            max_veh = max(max_veh, frame_veh)
            max_per = max(max_per, frame_per)

    except Exception as e:
        print(f"  YOLO error: {e}")

    return {
        "vehicles":          list(all_vehicles),
        "persons":           list(all_persons),
        "signs":             list(all_signs),
        "animals":           list(all_animals),
        "max_vehicle_count": max_veh,
        "max_person_count":  max_per,
        "has_traffic_light": has_tl,
        "has_stop_sign":     has_ss,
    }


print("   YOLO perception branch ready (batched + stable FP32 on cuda:0)")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading YOLOv8...
   YOLOv8 loaded on cuda:0 | classes: 80
   YOLO perception branch ready (batched + stable FP32 on cuda:0)


In [8]:
import torch
from PIL import Image
from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
    BitsAndBytesConfig
)

# ── 4-BIT CONFIG ──────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# ── LOAD MODEL SPLIT ACROSS BOTH GPUs ─────────────────────────
print("Loading Qwen2.5-VL-7B in 4-bit across both GPUs...")

VLM_MODEL = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",              # let accelerate split across cuda:0 + cuda:1
    max_memory={
        0: "13GiB",                 # leave 2GB headroom on each GPU
        1: "13GiB",
    },
    trust_remote_code=True,
)
VLM_MODEL.eval()

# ── PROCESSOR ─────────────────────────────────────────────────
VLM_PROCESSOR = AutoProcessor.from_pretrained(
    VLM_MODEL_NAME,
    trust_remote_code=True
)

# ── FIND WHERE THE FIRST LAYER LIVES (inputs must go here) ────
DEVICE = next(VLM_MODEL.parameters()).device
print(f"Input device: {DEVICE}")
print(f"Input device (embed_tokens): {DEVICE}")

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {free/1024**3:.1f}GB free / {total/1024**3:.1f}GB total")

print("✅ Qwen loaded successfully across both GPUs")

Loading Qwen2.5-VL-7B in 4-bit across both GPUs...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Input device: cuda:0
Input device (embed_tokens): cuda:0
GPU 0: 12.1GB free / 14.6GB total
GPU 1: 7.7GB free / 14.6GB total
✅ Qwen loaded successfully across both GPUs


In [9]:
import torch, json, re
from PIL import Image

# ─────────────────────────────────────────────────────────────
#  PASS 1 LEGEND — Scene / Environment (11 questions)
# ─────────────────────────────────────────────────────────────
LEGEND_SCENE = """
Q1a_time_of_day: Day=0, Dawn=1, Dusk=2, Night=3, Unknown=-1
Q1b_weather: Sunny=0, Cloudy=1, Rainy=2, PartlyCloudy=3, Snowy=4, Foggy=5, Unknown=-1
Q2_lighting: Daylight=0, HeadlightsOnly=1, SunriseSunset=2, Streetlights=4, Unknown=-1
Q3a_environment: Urban=0, Suburban=2, Rural=3, Unknown=-1
Q3b_facilities: UrbanSkyscrapers=1, ConstructionZone=2, ResidentialArea=3, ParkingLot=4, School=5, Unknown=-1
Q4a_road_config: Highway=1, Intersection=2, TJunction=3, ResidentialStreet=4, Bridge=5, Roundabout=6, Ramp=7, Unknown=-1
Q4b_lane_config: TwoWayDivided=0, TwoWayNotDivided=1, OneWay=2, Unknown=-1
Q4c_ego_lane: Right=0, Left=1, Middle=2, Unknown=-1
Q8_traffic_control: Yield=0, None=1, TrafficLight=3, Construction=4, NoEntry=5, PedestrianCrossing=6, TrafficLightAhead=7, SchoolZone=8, SpeedLimit=9, StopSign=10, Railroad=11, Unknown=-1
Q9a_road_surface: DryPavement=0, WetRoad=1, SnowIce=2, Debris=3, Unknown=-1
Q9b_road_material: Asphalt=0, Concrete=1, GravelDirt=3, Unknown=-1
"""

SCENE_FIELDS = [
    "Q1a_time_of_day","Q1b_weather","Q2_lighting",
    "Q3a_environment","Q3b_facilities",
    "Q4a_road_config","Q4b_lane_config","Q4c_ego_lane",
    "Q8_traffic_control","Q9a_road_surface","Q9b_road_material",
]
JSON_TEMPLATE_SCENE = {f: "<int>" for f in SCENE_FIELDS}

# ─────────────────────────────────────────────────────────────
#  PASS 2 LEGEND — Incident (14 questions)
# ─────────────────────────────────────────────────────────────
LEGEND_INCIDENT = """
Q5a_incident_cat: PedestrianVRU=0, VehicleToVehicle=1, AnimalsObjects=2, NoCollision=3, BarriersFixed=5, VehicleLossControl=6, Unknown=-1
Q5b_primary_entity: Pedestrian=0, EgoCar=1, Animal=2, NoCollision=3, AnotherVehicle=5, Cyclist=6, FlyingObject=10, Smoke=13, ObjectOnRoad=29, Unknown=-1
Q5c_primary_veh: SmallCar=0, NoCollision=1, Truck=2, Bicycle=3, BicycleCart=4, Motorcycle=5, Scooter=6, SUV=8, Bus=9, Unknown=-1
Q5e_primary_beh: Crossing=0, Speeding=1, DrivingNormally=3, OvertakingLanes=4, Turning=6, Walking=7, Stopped=8, Swerving=10, IgnoringSignal=11, LostControl=12, FellDown=13, RollingThrough=14, FailureToYield=15, Flying=16, RollingOver=17, Stationary=18, FallenIntoRoad=19, Unknown=-1
Q5f_secondary_entity: Cyclist=0, EgoCar=1, AnotherVehicle=2, ObjectBarrier=3, Pedestrian=4, Scooter=5, Animal=6, Unknown=-1
Q5g_secondary_veh: Scooter=0, SmallCar=1, Bicycle=2, Truck=3, SUV=4, Bus=5, Unknown=-1
Q5i_secondary_beh: DrivingNormally=0, Swerving=1, FixedPosition=2, Speeding=3, Crossing=4, Turning=5, LostControl=6, FailureToYield=7, Flying=8, Stopped=9, FallenOntoRoad=10, OvertakingLanes=11, RollingOver=13, UTurn=14, Unknown=-1
Q5j_incident_peak: NearMiss=0, NoCollision=1, Collision=2, NoEffect=3, AvoidedInAdvance=4, ChainReaction=5, Unknown=-1
Q6a_prev_primary: UnsafeTravel=0, NoFaultNormal=1, CouldHaveManeuvered=2, NoIncident=3, NoFaultAvoided=4, NotCapable=5, Defect=6, Unknown=-1
Q6b_prev_secondary: NoFaultAvoided=0, NoFaultNormal=1, NotCapable=2, UnsafeTravel=3, NoIncident=4, Defect=5, CouldHaveManeuvered=6, Unknown=-1
Q7a_primary_impact_loc: NoCollision=0, Front=1, LeftSide=2, RightSide=4, Rear=5, TopRoof=6, Unknown=-1
Q7b_primary_impact_side: NoCollision=0, LeftCorner=2, Front=3, RightSide=4, RightCorner=5, Undercarriage=6, LeftSide=7, Unknown=-1
Q7c_secondary_impact_loc: NoCollision=0, RightSide=1, Front=2, TopRoof=3, LeftSide=5, Rear=6, Unknown=-1
Q7d_secondary_impact_side: NoCollision=0, LeftCorner=1, RightCorner=3, LeftSide=4, RoofTop=5, RightSide=6, FrontGrill=7, MidRight=9, MidLeft=10, Undercarriage=11, Rear=12, Unknown=-1
"""

INCIDENT_FIELDS = [
    "Q5a_incident_cat","Q5b_primary_entity","Q5c_primary_veh",
    "Q5e_primary_beh","Q5f_secondary_entity","Q5g_secondary_veh",
    "Q5i_secondary_beh","Q5j_incident_peak",
    "Q6a_prev_primary","Q6b_prev_secondary",
    "Q7a_primary_impact_loc","Q7b_primary_impact_side",
    "Q7c_secondary_impact_loc","Q7d_secondary_impact_side",
]
JSON_TEMPLATE_INCIDENT = {f: "<int>" for f in INCIDENT_FIELDS}

# ─────────────────────────────────────────────────────────────
#  Keep the combined field mapping — used by validate & postproc
# ─────────────────────────────────────────────────────────────
FIELD_TO_QID = {
    "Q1a_time_of_day":"Q1.a","Q1b_weather":"Q1.b","Q2_lighting":"Q2",
    "Q3a_environment":"Q3.a","Q3b_facilities":"Q3.b",
    "Q4a_road_config":"Q4.a","Q4b_lane_config":"Q4.b","Q4c_ego_lane":"Q4.c",
    "Q5a_incident_cat":"Q5.a","Q5b_primary_entity":"Q5.b","Q5c_primary_veh":"Q5.c",
    "Q5e_primary_beh":"Q5.e","Q5f_secondary_entity":"Q5.f",
    "Q5g_secondary_veh":"Q5.g","Q5i_secondary_beh":"Q5.i",
    "Q5j_incident_peak":"Q5.j","Q6a_prev_primary":"Q6.a","Q6b_prev_secondary":"Q6.b",
    "Q7a_primary_impact_loc":"Q7.a","Q7b_primary_impact_side":"Q7.b",
    "Q7c_secondary_impact_loc":"Q7.c","Q7d_secondary_impact_side":"Q7.d",
    "Q8_traffic_control":"Q8","Q9a_road_surface":"Q9.a","Q9b_road_material":"Q9.b",
}
QID_TO_FIELD = {v: k for k, v in FIELD_TO_QID.items()}


# ─────────────────────────────────────────────────────────────
#  PROMPT BUILDERS
# ─────────────────────────────────────────────────────────────

def _build_scene_prompt(yolo: dict) -> str:
    signs   = ", ".join(yolo.get("signs", [])) or "none"
    vehicles = ", ".join(yolo.get("vehicles", [])) or "none"

    return f"""You are a traffic scene classifier. Look at the video frames carefully.

[Detected by object detector]
Vehicles seen: {vehicles}
Signs/signals seen: {signs}

Focus ONLY on: time of day, weather, lighting, road type, lane config, traffic signs, road surface.
Do NOT classify the incident yet — only the scene environment.

STRICT RULES:
- Output ONLY valid JSON, nothing else
- All values must be integers from the legend
- All {len(SCENE_FIELDS)} fields must be present

LEGEND:
{LEGEND_SCENE}

OUTPUT FORMAT:
{json.dumps(JSON_TEMPLATE_SCENE)}
"""


def _build_incident_prompt(yolo: dict, temporal: dict, scene_context: str) -> str:
    vehicles = ", ".join(yolo.get("vehicles", [])) or "none"
    persons  = len(yolo.get("persons", []))
    animals  = ", ".join(yolo.get("animals", [])) or "none"
    motion   = temporal.get("motion_ratio", 0)
    phase    = temporal.get("peak_phase", "unknown")
    is_coll  = temporal.get("is_collision", False)

    return f"""You are a traffic incident classifier. Look at the video frames carefully.

[Scene context already identified]
{scene_context}

[Detected by object detector]
Vehicles: {vehicles}
Persons: {persons}
Animals: {animals}

[Motion analysis]
Motion level: {motion:.1f}  (>10 = high motion, likely collision)
Peak phase: {phase}
Collision detected by motion: {is_coll}

Guidelines:
- High motion in incident phase → likely Collision (Q5j=2)
- Low motion throughout → likely NoCollision (Q5j=1)
- Use detected objects as primary evidence for entity types
- Be consistent: if no collision then Q7a/b/c/d should all be 0

STRICT RULES:
- Output ONLY valid JSON, nothing else
- All values must be integers from the legend
- All {len(INCIDENT_FIELDS)} fields must be present

LEGEND:
{LEGEND_INCIDENT}

OUTPUT FORMAT:
{json.dumps(JSON_TEMPLATE_INCIDENT)}
"""


# ─────────────────────────────────────────────────────────────
#  JSON PARSER  (unchanged)
# ─────────────────────────────────────────────────────────────

def _parse_json_safe(text: str) -> dict:
    if not text:
        return None
    try:
        return json.loads(text.strip())
    except:
        pass
    try:
        s, e = text.index("{"), text.rindex("}") + 1
        return json.loads(text[s:e])
    except:
        pass
    try:
        cleaned = re.sub(r"```(?:json)?", "", text).strip("`").strip()
        return json.loads(cleaned)
    except:
        pass
    try:
        pairs = re.findall(r'"(Q\w+)"\s*:\s*(-?\d+)', text)
        if pairs:
            return {k: int(v) for k, v in pairs}
    except:
        pass
    return None


# ─────────────────────────────────────────────────────────────
#  VALIDATION  (unchanged)
# ─────────────────────────────────────────────────────────────

def _validate_output(parsed: dict) -> dict:
    result = {qid: -1 for qid in LABEL_MAP}
    if not parsed:
        return result
    for field, val in parsed.items():
        qid = FIELD_TO_QID.get(field)
        if qid is None:
            continue
        try:
            iv = int(val)
        except:
            continue
        valid = VALID_VALUES.get(qid, set())
        result[qid] = iv if (not valid or iv in valid) else -1
    return result


# ─────────────────────────────────────────────────────────────
#  CORE INFERENCE HELPER
# ─────────────────────────────────────────────────────────────

def _infer(img_pil: Image.Image, prompt: str, max_tokens: int) -> str:
    from qwen_vl_utils import process_vision_info
    content = [{"type": "image", "image": img_pil},
               {"type": "text",  "text": prompt}]
    messages = [{"role": "user", "content": content}]

    text_in = VLM_PROCESSOR.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = VLM_PROCESSOR(
        text=[text_in],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.inference_mode():
        out_ids = VLM_MODEL.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=VLM_PROCESSOR.tokenizer.eos_token_id,
        )
    new_tokens = out_ids[:, inputs["input_ids"].shape[1]:]
    return VLM_PROCESSOR.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()


# ─────────────────────────────────────────────────────────────
#  TWO-PASS run_qwen  — drop-in replacement for old run_qwen
# ─────────────────────────────────────────────────────────────

SCENE_MAX_TOKENS    = 160   # 11 fields × ~12 tokens each + structure
INCIDENT_MAX_TOKENS = 220   # 14 fields × ~12 tokens each + structure

def run_qwen(grid_bgr: np.ndarray, yolo: dict, temporal: dict) -> dict:
    fallback = {qid: -1 for qid in LABEL_MAP}

    grid_rgb = cv2.cvtColor(grid_bgr, cv2.COLOR_BGR2RGB)
    img_pil  = Image.fromarray(grid_rgb)

    # ── PASS 1: Scene questions ───────────────────────────────
    scene_prompt = _build_scene_prompt(yolo)
    try:
        raw1   = _infer(img_pil, scene_prompt, SCENE_MAX_TOKENS)
        scene_parsed = _parse_json_safe(raw1)
        if scene_parsed is None:
            smaller = img_pil.resize((img_pil.width // 2, img_pil.height // 2))
            raw1 = _infer(smaller, scene_prompt, SCENE_MAX_TOKENS)
            scene_parsed = _parse_json_safe(raw1)
    except Exception as e:
        print(f"Qwen Pass1 error: {e}")
        scene_parsed = None

    scene_result = _validate_output(scene_parsed) if scene_parsed else fallback.copy()

    # Build a compact human-readable scene summary to feed into Pass 2
    tod_map  = {0:"Day", 1:"Dawn", 2:"Dusk", 3:"Night"}
    env_map  = {0:"Urban", 2:"Suburban", 3:"Rural"}
    road_map = {1:"Highway", 2:"Intersection", 3:"T-junction", 4:"ResidentialStreet",
                5:"Bridge", 6:"Roundabout", 7:"Ramp"}

    scene_context = (
        f"Time of day: {tod_map.get(scene_result.get('Q1.a'), 'Unknown')} | "
        f"Weather: {scene_result.get('Q1.b', -1)} | "
        f"Lighting: {scene_result.get('Q2', -1)} | "
        f"Environment: {env_map.get(scene_result.get('Q3.a'), 'Unknown')} | "
        f"Road: {road_map.get(scene_result.get('Q4.a'), 'Unknown')} | "
        f"Lane config: {scene_result.get('Q4.b', -1)} | "
        f"Traffic control: {scene_result.get('Q8', -1)}"
    )

    # ── PASS 2: Incident questions (grounded by Pass 1 context) ──
    incident_prompt = _build_incident_prompt(yolo, temporal, scene_context)
    try:
        raw2 = _infer(img_pil, incident_prompt, INCIDENT_MAX_TOKENS)
        incident_parsed = _parse_json_safe(raw2)
        if incident_parsed is None:
            smaller = img_pil.resize((img_pil.width // 2, img_pil.height // 2))
            raw2 = _infer(smaller, incident_prompt, INCIDENT_MAX_TOKENS)
            incident_parsed = _parse_json_safe(raw2)
    except Exception as e:
        print(f"Qwen Pass2 error: {e}")
        incident_parsed = None

    incident_result = _validate_output(incident_parsed) if incident_parsed else fallback.copy()

    # ── MERGE: scene wins for scene QIDs, incident wins for incident QIDs ──
    merged = {}
    for qid in LABEL_MAP:
        field = QID_TO_FIELD.get(qid)
        if field in SCENE_FIELDS:
            merged[qid] = scene_result.get(qid, -1)
        else:
            merged[qid] = incident_result.get(qid, -1)

    return merged


print("✅ Two-pass Qwen pipeline ready")
print(f"   Pass 1 — {len(SCENE_FIELDS)} scene questions   (max {SCENE_MAX_TOKENS} tokens)")
print(f"   Pass 2 — {len(INCIDENT_FIELDS)} incident questions (max {INCIDENT_MAX_TOKENS} tokens)")
print("   Pass 1 result fed as context into Pass 2 prompt")

✅ Two-pass Qwen pipeline ready
   Pass 1 — 11 scene questions   (max 160 tokens)
   Pass 2 — 14 incident questions (max 220 tokens)
   Pass 1 result fed as context into Pass 2 prompt


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# POST-PROCESSING — FINAL OPTIMIZED VERSION
# Priority order: YOLO hard overrides → Temporal → YOLO fallback → Smart defaults → Consistency → Validation
# ─────────────────────────────────────────────────────────────────────────────

SMART_DEFAULTS = {
    "Q1.a": 0,   # Day
    "Q1.b": 0,   # Sunny/Clear
    "Q2"  : 0,   # Daylight
    "Q3.a": 0,   # Urban
    "Q3.b": 1,   # Urban facilities
    "Q4.a": 2,   # Intersection
    "Q4.b": 1,   # Two-way not divided
    "Q4.c": 0,   # Right lane
    "Q5.e": 3,   # Driving normally
    "Q5.i": 0,   # Driving normally
    "Q6.a": 2,   # Could have maneuvered
    "Q6.b": 1,   # No fault, behaved normally
    "Q7.a": 0,   # No collision
    "Q7.b": 0,   # No collision
    "Q7.c": 0,   # No collision
    "Q7.d": 0,   # No collision
    "Q8"  : 1,   # None visible
    "Q9.a": 0,   # Dry pavement
    "Q9.b": 0,   # Asphalt
}

LOW_MOTION  = 2.0
MED_MOTION  = 5.0
HIGH_MOTION = 10.0

# Threshold: if Qwen returns more than this many unknowns → low confidence
LOW_CONFIDENCE_THRESHOLD = 8


def apply_postprocessing(preds: dict, yolo: dict, temporal: dict) -> dict:
    p = dict(preds)

    yolo_failed     = yolo.get("yolo_failed", False)
    v               = yolo.get("vehicles", [])
    motion          = temporal.get("motion_ratio", 0.0)
    incident_motion = temporal.get("motion_incident", 0.0)
    peak_phase      = temporal.get("peak_phase", "unknown")
    is_collision    = temporal.get("is_collision", False)

    unknown_count  = sum(1 for val in p.values() if val == -1)
    low_confidence = unknown_count > LOW_CONFIDENCE_THRESHOLD

    # ── LAYER 0: Q8 — YOLO wins when confident, preserves Qwen otherwise ─────
    if not yolo_failed:
        if yolo.get("has_traffic_light"):
            p["Q8"] = 3
        elif yolo.get("has_stop_sign"):
            p["Q8"] = 10
        elif p.get("Q8", -1) == -1 or low_confidence:
            p["Q8"] = 1    # None visible — only when Qwen is weak/unknown
        elif low_confidence and p.get("Q8") not in (3, 10):
            p["Q8"] = 1

    # ── LAYER 1: YOLO entity fallbacks ───────────────────────────────────────
    if not yolo_failed:

        if p.get("Q5.a", -1) == -1 or low_confidence:
            if yolo["max_person_count"] > 0 and yolo["max_vehicle_count"] == 0:
                p["Q5.a"] = 0
            elif yolo["max_vehicle_count"] >= 2 and motion > LOW_MOTION:
                p["Q5.a"] = 1
            elif yolo.get("animals"):
                p["Q5.a"] = 2
            else:
                p["Q5.a"] = 3

        if p.get("Q5.b", -1) == -1 or low_confidence:
            if yolo["max_person_count"] > 0:
                p["Q5.b"] = 0
            elif yolo["max_vehicle_count"] >= 1:
                p["Q5.b"] = 5
            elif yolo.get("animals"):
                p["Q5.b"] = 2
            else:
                p["Q5.b"] = 3

        if p.get("Q5.c", -1) == -1:
            if "truck"        in v: p["Q5.c"] = 2
            elif "bus"        in v: p["Q5.c"] = 9
            elif "motorcycle" in v: p["Q5.c"] = 5
            elif "bicycle"    in v: p["Q5.c"] = 3
            elif "car"        in v: p["Q5.c"] = 0
            else:                   p["Q5.c"] = 1

        if p.get("Q5.f", -1) == -1:
            if yolo["max_vehicle_count"] >= 2:
                p["Q5.f"] = 2
            elif yolo["max_person_count"] > 0 and yolo["max_vehicle_count"] >= 1:
                p["Q5.f"] = 4
            else:
                p["Q5.f"] = 1

        if p.get("Q5.g", -1) == -1:
            if yolo["max_vehicle_count"] >= 2:
                if "truck" in v:  p["Q5.g"] = 3
                elif "bus"  in v: p["Q5.g"] = 5
                elif "car"  in v: p["Q5.g"] = 1
                else:             p["Q5.g"] = 1
            else:
                p["Q5.g"] = 1

    # ── LAYER 2: Temporal fallbacks ───────────────────────────────────────────
    if p.get("Q5.j", -1) == -1 or low_confidence:
        if is_collision or (incident_motion > HIGH_MOTION and peak_phase == "incident"):
            p["Q5.j"] = 2
        elif motion > MED_MOTION:
            p["Q5.j"] = 0
        else:
            p["Q5.j"] = 1

    if p.get("Q5.e", -1) == -1 and motion > MED_MOTION:
        p["Q5.e"] = 1

    if p.get("Q5.b") == 2:
        p["Q6.a"] = 5
    elif p.get("Q6.a", -1) == -1 and is_collision:
        p["Q6.a"] = 2

    # ── LAYER 3: Q7 impact location — handles -1 AND low_confidence ──────────
    q5j = p.get("Q5.j", -1)

    if q5j == 1:
        # No collision → all impact fields must be 0
        p["Q7.a"] = 0
        p["Q7.b"] = 0
        p["Q7.c"] = 0
        p["Q7.d"] = 0
    else:
        # Vehicle-based heuristic — triggers on -1 OR low confidence
        if not yolo_failed and yolo["max_vehicle_count"] >= 2:
            if p.get("Q7.a", -1) == -1 or low_confidence:
                p["Q7.a"] = 1   # Front impact
            if p.get("Q7.b", -1) == -1 or low_confidence:
                p["Q7.b"] = 3   # Front grill/bumper

        # Guarantee no -1 remains in any Q7 field
        for q in ["Q7.a", "Q7.b", "Q7.c", "Q7.d"]:
            if p.get(q, -1) == -1:
                p[q] = 0

    # ── LAYER 4: Smart defaults — remaining -1s only ──────────────────────────
    for qid, default_val in SMART_DEFAULTS.items():
        if p.get(qid, -1) == -1:
            p[qid] = default_val

    # ── LAYER 5: Consistency rules ────────────────────────────────────────────
    if (p.get("Q5.j") == 1 or p.get("Q5.a") == 3) and motion < LOW_MOTION:
        p["Q5.a"] = 3
        p["Q5.j"] = 1
        p["Q7.a"] = 0
        p["Q7.b"] = 0
        p["Q7.c"] = 0
        p["Q7.d"] = 0
        p["Q6.a"] = 3
        p["Q6.b"] = 4

    VEHICLE_SECONDARY = {1, 2}  # ego-car, another vehicle
    if p.get("Q5.f") not in VEHICLE_SECONDARY:
        p["Q5.g"] = -2
        p["Q7.c"] = -2
        p["Q7.d"] = -2

    VEHICLE_PRIMARY = {1, 5}  # ego-car, another vehicle
    if p.get("Q5.b") not in VEHICLE_PRIMARY:
        p["Q5.c"] = -2
        p["Q7.a"] = -2
        p["Q7.b"] = -2

    if p.get("Q1.a") == 3 and p.get("Q2") == 0:
        p["Q2"] = 1

    if p.get("Q1.a") in (1, 2) and p.get("Q2") == 0:
        p["Q2"] = 2

    if motion < LOW_MOTION and incident_motion < MED_MOTION and p.get("Q5.j") == 2:
        p["Q5.j"] = 0

    # ── LAYER 6: Final validation ─────────────────────────────────────────────
    for qid in LABEL_MAP:
        val   = p.get(qid, -1)
        valid = VALID_VALUES.get(qid, set())
        if valid and val not in valid:
            p[qid] = SMART_DEFAULTS.get(qid, -1)
        try:
            p[qid] = int(p[qid])
        except:
            p[qid] = SMART_DEFAULTS.get(qid, -1)

    return p


print("✅ Post-processing ready")
print("   Layer 0 : YOLO hard overrides (Q8 always set — never -1)")
print("   Layer 1 : YOLO entity fallbacks (fires on -1 OR low confidence)")
print("   Layer 2 : Temporal fallbacks (Q5.j, Q5.e, Q6.a)")
print("   Layer 3 : Q7 impact location (no-collision + vehicle heuristic)")
print("   Layer 4 : Smart defaults (last resort)")
print("   Layer 5 : Consistency rules (7 rules, no destructive overrides)")
print("   Layer 6 : Final validation (type safety + valid set check)")

✅ Post-processing ready
   Layer 0 : YOLO hard overrides (Q8 always set — never -1)
   Layer 1 : YOLO entity fallbacks (fires on -1 OR low confidence)
   Layer 2 : Temporal fallbacks (Q5.j, Q5.e, Q6.a)
   Layer 3 : Q7 impact location (no-collision + vehicle heuristic)
   Layer 4 : Smart defaults (last resort)
   Layer 5 : Consistency rules (7 rules, no destructive overrides)
   Layer 6 : Final validation (type safety + valid set check)


In [11]:
import os, json, time, threading
import torch, cv2, numpy as np
from PIL import Image
from tqdm.auto import tqdm


def process_video(video_path: str, video_filename: str) -> dict:
    """
    Fully corrected pipeline:
    - Sequential YOLO → Qwen (correct fusion)
    - Robust fallbacks
    - No misleading defaults
    - Cleaner GPU handling
    """

    fallback = {qid: -1 for qid in LABEL_MAP}

    # ── Stage 1: Frame Sampling ──────────────────────────────────────────────
    try:
        frames, meta = sample_frames(video_path)
        if not frames:
            return fallback
        if meta.get("warnings"):
            print(f"    sampling warn: {meta['warnings']}")
    except Exception as e:
        print(f"    Frame sampling failed: {e}")
        return fallback

    # ── Stage 2: Temporal Features (CPU) ─────────────────────────────────────
    try:
        temporal_feats = extract_temporal_features(frames)
    except Exception as e:
        print(f"    Temporal failed: {e}")
        temporal_feats = {
            "motion_ratio": 1.0,
            "is_collision": False,
            "peak_phase": "unknown",
            "intensity": "low",
            "motion_before": 0,
            "motion_incident": 0,
            "motion_after": 0
        }

    # ── Stage 3: Build Grid for VLM ─────────────────────────────────────────
    try:
        grid_bgr = build_grid(frames, temporal_feats, GRID_ROWS, GRID_COLS, FRAME_SIZE)
    except Exception as e:
        print(f"    Grid build failed: {e}")
        return fallback

    # ── Stage 4: YOLO Detection (RUN FIRST, CRITICAL) ────────────────────────
    try:
        torch.cuda.set_device(0)
        yolo_result = aggregate_detections(frames)

        # Add explicit success flag
        yolo_result["yolo_failed"] = False

    except Exception as e:
        print(f"    YOLO error: {e}")

        yolo_result = {
            "vehicles": [],
            "persons": [],
            "signs": [],
            "animals": [],
            "max_vehicle_count": 0,
            "max_person_count": 0,
            "has_traffic_light": False,
            "has_stop_sign": False,
            "yolo_failed": True
        }

    # ── Stage 5: Qwen VLM (NOW USE YOLO + TEMPORAL) ─────────────────────────
    try:
        torch.cuda.set_device(0)  # keep same GPU for stability

        qwen_result = run_qwen(
            grid_bgr,
            yolo_result,      #   FIX: pass YOLO features
            temporal_feats
        )

        # Ensure valid structure
        if not isinstance(qwen_result, dict):
            raise ValueError("Invalid Qwen output")

    except Exception as e:
        print(f"    Qwen error: {e}")

        # Partial fallback instead of full wipe
        qwen_result = {}
        for qid in LABEL_MAP:
            qwen_result[qid] = -1

    # ── Stage 6: Post-processing (SAFE + CONTROLLED) ────────────────────────
    try:
        final = apply_postprocessing(
            qwen_result,
            yolo_result,
            temporal_feats
        )
    except Exception as e:
        print(f"    Post-process failed: {e}")
        final = qwen_result  # fallback to raw Qwen

    return final


# ─────────────────────────────────────────────────────────────────────────────
# CHECKPOINT SYSTEM
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd, glob

def save_checkpoint(all_preds: dict, done_count: int):
    """Save current predictions as both JSON and CSV checkpoint."""
    ckpt_path = f"{CHECKPOINT_DIR}/checkpoint_{done_count}.json"
    with open(ckpt_path, "w") as f:
        json.dump(all_preds, f)
    print(f"     Checkpoint saved: {ckpt_path}  ({done_count} videos)")


def load_checkpoint() -> dict:
    """Load latest checkpoint if exists. Returns (preds_dict, completed_ids)."""
    checkpoints = sorted(glob.glob(f"{CHECKPOINT_DIR}/checkpoint_*.json"))
    if not checkpoints:
        print("No checkpoint found → starting fresh")
        return {}, []
    latest = checkpoints[-1]
    with open(latest) as f:
        preds = json.load(f)
    completed = list(preds.keys())
    print(f"   Resumed from {latest} ({len(completed)} videos done)")
    return preds, completed


# ─────────────────────────────────────────────────────────────────────────────
# BATCH LOOP
# ─────────────────────────────────────────────────────────────────────────────

all_videos = sorted(
    [f for f in os.listdir(VIDEO_DIR)
     if f.lower().endswith((".mp4",".avi",".mov"))],
    key=lambda x: int(x.split(".")[0]) if x.split(".")[0].isdigit() else 0
)

if VIDEO_ID_RANGE is not None:
    s, e = VIDEO_ID_RANGE
    all_videos = [v for v in all_videos
                  if int(v.split(".")[0]) >= s and int(v.split(".")[0]) < e]

# Resume from checkpoint
all_preds, completed_ids = load_checkpoint()
completed_set = set(str(c) for c in completed_ids)
remaining     = [v for v in all_videos
                 if v.split(".")[0] not in completed_set]

print(f"Total videos  : {len(all_videos)}")
print(f"Already done  : {len(completed_ids)}")
print(f"Remaining     : {len(remaining)}")

errors = {}

for i, vid_file in enumerate(tqdm(remaining, desc="Processing")):
    vid_id   = vid_file.split(".")[0]
    vid_path = os.path.join(VIDEO_DIR, vid_file)
    t0       = time.time()

    if not os.path.exists(vid_path):
        print(f"      {vid_file}: FILE NOT FOUND")
        all_preds[vid_id] = {qid: -1 for qid in LABEL_MAP}
        errors[vid_id]    = "file not found"
        continue

    try:
        preds = process_video(vid_path, vid_file)
        all_preds[vid_id] = preds
        elapsed = time.time() - t0
        done    = len(completed_ids) + i + 1
        eta_min = (len(remaining) - i - 1) * elapsed / 60
        tqdm.write(f"  ✓ {vid_file} | {elapsed:.1f}s | ETA ~{eta_min:.0f}min")
    except Exception as ex:
        errors[vid_id]    = str(ex)
        all_preds[vid_id] = {qid: -1 for qid in LABEL_MAP}
        tqdm.write(f"      {vid_file}: {ex}")

    # Checkpoint every N videos
    if (i + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(all_preds, len(completed_ids) + i + 1)

print(f"\n   Done | success={len(all_preds)-len(errors)} | errors={len(errors)}")


No checkpoint found → starting fresh
Total videos  : 661
Already done  : 0
Remaining     : 661


Processing:   0%|          | 0/661 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  ✓ 0.mp4 | 41.4s | ETA ~455min
  ✓ 1.mp4 | 38.2s | ETA ~420min
    sampling warn: ['4 frames → padding']
  ✓ 2.mp4 | 37.9s | ETA ~415min
    sampling warn: ['3 frames → padding']
  ✓ 3.mp4 | 37.2s | ETA ~407min
    sampling warn: ['7 frames → padding']
  ✓ 4.mp4 | 37.5s | ETA ~410min
    sampling warn: ['3 frames → padding']
  ✓ 5.mp4 | 38.1s | ETA ~416min
    sampling warn: ['3 frames → padding']
  ✓ 6.mp4 | 38.2s | ETA ~416min
    sampling warn: ['4 frames → padding']
  ✓ 7.mp4 | 38.4s | ETA ~418min
    sampling warn: ['3 frames → padding']
  ✓ 8.mp4 | 38.4s | ETA ~417min
    sampling warn: ['6 frames → padding']
  ✓ 9.mp4 | 38.4s | ETA ~416min
    sampling warn: ['5 frames → padding']
  ✓ 10.mp4 | 38.2s | ETA ~413min
    sampling warn: ['4 frames → padding']
  ✓ 11.mp4 | 39.0s | ETA ~422min
    sampling warn: ['3 frames → padding']
  ✓ 12.mp4 | 38.4s | ETA ~415min
    sampling warn: ['4 frames → padding']
  ✓ 13.mp4 | 38.2s | ETA ~412min
    sampling warn: ['5 frames → padding']
  

In [12]:
import json

with open(SUBMISSION_JSON, "w") as f:
    json.dump(all_preds, f, indent=2)

print(f"   Saved → {SUBMISSION_JSON}  ({len(all_preds)} videos)")

# Sanity check on one sample
sample_vid = list(all_preds.keys())[0]
print(f"\nSample (video_id={sample_vid}):")
for qid, val in all_preds[sample_vid].items():
    rev = {v:k for k,v in LABEL_MAP[qid]["answers"].items()}
    print(f"  {qid:8s} = {val:3d}  ({rev.get(val,'?')})")


   Saved → /kaggle/working/submission.json  (661 videos)

Sample (video_id=0):
  Q1.a     =   3  (Night)
  Q1.b     =   0  (Sunny/Clear)
  Q2       =   4  (Streetlights on)
  Q3.a     =   0  (Urban area – City streets with dense buildings, intersections, and traffic)
  Q3.b     =   3  (Residential Area)
  Q4.a     =   4  (Residential street)
  Q4.b     =   1  (Two-way not divided)
  Q4.c     =   2  (Middle)
  Q5.a     =   3  (No Collision)
  Q5.b     =   3  (No collision)
  Q5.c     =  -1  (Unknown)
  Q5.e     =   8  (Stopped)
  Q5.f     =   2  (Another Vehicle)
  Q5.g     =   3  (Truck)
  Q5.i     =   9  (Stopped)
  Q5.j     =   1  (No collision)
  Q6.a     =   3  (No Incident)
  Q6.b     =   4  (No Incident)
  Q7.a     =   0  (No collision)
  Q7.b     =   0  (No collision)
  Q7.c     =   0  (No Collision)
  Q7.d     =   0  (No collision)
  Q8       =   3  (Traffic light)
  Q9.a     =   0  (Dry pavement)
  Q9.b     =   0  (Asphalt / Blacktop)


In [13]:
import pandas as pd
import json

with open(SUBMISSION_JSON, "r") as f:
    data_json = json.load(f)

df_template = pd.read_csv(SAMPLE_CSV_PATH)
columns     = df_template.columns

final_rows = []
for i in range(661):
    vid_str    = str(i)
    row        = {"video_id": i}
    video_data = data_json.get(vid_str, {})

    for col in columns:
        if col == "video_id": continue
        found = False
        for q_key in video_data:
            if f"{q_key}:" in col or f"{q_key} " in col:
                row[col] = int(video_data[q_key])
                found = True
                break
        if not found:
            row[col] = -1

    final_rows.append(row)

df_final = pd.DataFrame(final_rows)[columns]
df_final["video_id"] = df_final["video_id"].astype(int)
df_final.to_csv(SUBMISSION_CSV, index=False)

print(f"   Saved → {SUBMISSION_CSV}")
print(f"   Rows    : {len(df_final)}")
print(f"   Columns : {len(df_final.columns)}")
df_final.head(3)


   Saved → /kaggle/working/submission_f.csv
   Rows    : 661
   Columns : 26


,video_id,A)Weather:\nQ1.a: What was the time of day during the incident?,A)Weather:\nQ1.b: What were the weather conditions during the incident?,B) Light Condition:\nQ2: What was the lighting condition at the time of the incident?,C) Traffic Environment:\nQ3.a: Where did the incident take place (traffic environment)?,C) Traffic Environment:\nQ3.b: What facilities were present in the surrounding environment?,D) Road Configuration\nQ4.a: What is the road configuration at the incident time?,D) Road Configuration\nQ4.b: Road lane configuration:,D) Road Configuration\nQ4.c: Lane direction of Ego-Car at time of incident:,E) Incident Type\nQ5.a: What is the category of the incident?,...,E) Incident Type\nQ5.j: Incident peak,F) Incident Prevention Measure\nQ6.a: How could this Incident most likely have been prevented by the Primary Entity?,F) Incident Prevention Measure\nQ6.b: How could this incident most likely have been prevented by the Secondary Entity?,"G)Initial Point of Impact\nQ7.a: If the Primary Entity was a vehicle, where on the vehicle did the initial impact occur?","G)Initial Point of Impact\nQ7.b: If the Primary Entity was a vehicle, what side of the impacted area was hit?","G)Initial Point of Impact\nQ7.c: If the Secondary Entity was a vehicle, where on the vehicle did the initial impact occur?","G)Initial Point of Impact\nQ7.d: If the Secondary Entity was a vehicle, what side of the impacted area was hit?",H) Traffic Control & Signage Presence:\nQ8: What traffic control or road sign is present at the incident location?,I) Road Surface & Material Condition\nQ9.a: What was the road surface condition during the incident?,I) Road Surface & Material Condition\nQ9.b: What type of road material is the vehicle driving on?
0,0,3,0,4,0,3,4,1,2,3,...,1,3,4,0,0,0,0,3,0,0
1,1,0,1,1,3,1,1,1,2,3,...,1,3,4,0,0,0,0,10,0,0
2,2,0,1,0,2,1,1,0,2,3,...,1,3,4,0,0,0,0,10,0,0


In [14]:
import pandas as pd

df = pd.read_csv(SUBMISSION_CSV)
print(f"Shape            : {df.shape}")
print(f"video_id dtype   : {df['video_id'].dtype}")
print(f"Expected rows    : 661")
print(f"Null values      : {df.isnull().sum().sum()}")

minus_one_pct = (df == -1).sum() / len(df) * 100
print("\n── Unknown rate per question ──")
for col, pct in minus_one_pct.items():
    if col == "video_id": continue
    bar  = "█" * int(pct/5) + "░" * (20 - int(pct/5))
    flag = "   " if pct > 30 else "   " if pct < 5 else ""
    print(f"  {col[:30]:30s} {bar} {pct:5.1f}%{flag}")

overall_unk = minus_one_pct.drop("video_id", errors="ignore").mean()
print(f"\nOverall unknown rate : {overall_unk:.1f}%")
print(f"{'   HEALTHY' if overall_unk < 15 else '    NEEDS ATTENTION' if overall_unk < 40 else '🔴 CRITICAL'}")
print(f"\n   Submission ready for upload!")
if errors:
    print(f"    {len(errors)} videos had errors")


Shape            : (661, 26)
video_id dtype   : int64
Expected rows    : 661
Null values      : 0

── Unknown rate per question ──
  A)Weather:
Q1.a: What was the  ░░░░░░░░░░░░░░░░░░░░   0.0%   
  A)Weather:
Q1.b: What were the ░░░░░░░░░░░░░░░░░░░░   0.0%   
  B) Light Condition:
Q2: What w ░░░░░░░░░░░░░░░░░░░░   0.0%   
  C) Traffic Environment:
Q3.a:  ░░░░░░░░░░░░░░░░░░░░   0.0%   
  C) Traffic Environment:
Q3.b:  ░░░░░░░░░░░░░░░░░░░░   0.0%   
  D) Road Configuration
Q4.a: Wh ░░░░░░░░░░░░░░░░░░░░   0.0%   
  D) Road Configuration
Q4.b: Ro ░░░░░░░░░░░░░░░░░░░░   0.0%   
  D) Road Configuration
Q4.c: La ░░░░░░░░░░░░░░░░░░░░   0.0%   
  E) Incident Type
Q5.a: What is ░░░░░░░░░░░░░░░░░░░░   0.0%   
  E) Incident Type
Q5.b: Primary ░░░░░░░░░░░░░░░░░░░░   0.0%   
  E) Incident Type
Q5.c: If Prim ████████████░░░░░░░░  61.3%   
  E) Incident Type
Q5.e: Primary ░░░░░░░░░░░░░░░░░░░░   0.0%   
  E) Incident Type
Q5.f Secondar ░░░░░░░░░░░░░░░░░░░░   0.0%   
  E) Incident Type:
Q5.g: If Sec ████

In [15]:
import os

print("Files in /kaggle/working/:")
for f in sorted(os.listdir("/kaggle/working/")):
    size = os.path.getsize(f"/kaggle/working/{f}")
    print(f"  {f:45s} {size/1024:.1f} KB")


Files in /kaggle/working/:
  =4.45.0                                       0.0 KB
  __notebook__.ipynb                            278.2 KB
  checkpoint_100.json                           27.4 KB
  checkpoint_150.json                           41.1 KB
  checkpoint_200.json                           54.8 KB
  checkpoint_250.json                           68.5 KB
  checkpoint_300.json                           82.2 KB
  checkpoint_350.json                           96.0 KB
  checkpoint_400.json                           109.7 KB
  checkpoint_450.json                           123.4 KB
  checkpoint_50.json                            13.7 KB
  checkpoint_500.json                           137.1 KB
  checkpoint_550.json                           150.9 KB
  checkpoint_600.json                           164.6 KB
  checkpoint_650.json                           178.3 KB
  submission.json                               249.8 KB
  submission_f.csv                              37.7 KB
  yolov8m.pt  